# B3b Defense – 04 XGBoost Training

**Objective:** Train XGBoost on `defense_feature_matrix.csv` using `TimeSeriesSplit(n_splits=5)`.
Report MAE, RMSE, sMAPE, and WMAPE per fold. Save the best model to disk.

In [8]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [9]:
DATA_PROCESSED = '../data/processed/'
MODELS_DIR     = '../models/'

os.makedirs(MODELS_DIR, exist_ok=True)

In [10]:
# Load feature matrix — explicit to_datetime() to avoid slow dateutil fallback in pandas 2.x
df = pd.read_csv(DATA_PROCESSED + 'defense_feature_matrix.csv', index_col='date')
df.index = pd.to_datetime(df.index)

# Define feature columns and target.
# ADEFNO is explicitly retained as a feature (leading indicator with lags 1–24).
# Only FDEFX itself is excluded from features to avoid target leakage.
feature_cols = [col for col in df.columns if col != 'FDEFX']
target_col   = 'FDEFX'

X = df[feature_cols]
y = df[target_col]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape:         {y.shape}')
print(f'Target col:           {target_col}')
print(f'FDEFX in feature_cols: {"FDEFX" in feature_cols}  (must be False)')
print(f'ADEFNO in feature_cols: {"ADEFNO" in feature_cols}  (must be True)')

Feature matrix shape: (287, 32)
Target shape:         (287,)
Target col:           FDEFX
FDEFX in feature_cols: False  (must be False)
ADEFNO in feature_cols: True  (must be True)


In [11]:
def smape(actual, pred):
    """Symmetric Mean Absolute Percentage Error (in %)"""
    numerator   = 2 * np.abs(actual - pred)
    denominator = np.abs(actual) + np.abs(pred)
    # Avoid division by zero
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask]) * 100


def wmape(actual, pred):
    """Weighted Mean Absolute Percentage Error (in %)"""
    return np.sum(np.abs(actual - pred)) / np.sum(np.abs(actual)) * 100

In [12]:
tscv = TimeSeriesSplit(n_splits=5)

fold_results = []
best_model   = None

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBRegressor(
        n_estimators     = 200,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.6,   # column subsampling reduces overfitting with many lag features
        reg_alpha        = 0.1,   # L1 regularisation drives near-zero feature weights to zero
        random_state     = 42,
        verbosity        = 0
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    smape_val = smape(y_test.values, y_pred)
    wmape_val = wmape(y_test.values, y_pred)

    fold_results.append({
        'Fold':  fold_idx,
        'MAE':   round(mae, 4),
        'RMSE':  round(rmse, 4),
        'sMAPE': round(smape_val, 2),
        'WMAPE': round(wmape_val, 2)
    })

    print(f'Fold {fold_idx}: MAE={mae:.4f}  RMSE={rmse:.4f}  sMAPE={smape_val:.2f}%  WMAPE={wmape_val:.2f}%')

    # Keep the last fold's model as the final model
    best_model = model

print('\nCross-validation complete.')

Fold 1: MAE=98.0858  RMSE=115.2675  sMAPE=14.00%  WMAPE=13.41%
Fold 2: MAE=12.5276  RMSE=15.0148  sMAPE=1.57%  WMAPE=1.56%
Fold 3: MAE=19.6076  RMSE=22.3750  sMAPE=2.68%  WMAPE=2.65%
Fold 4: MAE=39.0137  RMSE=48.6436  sMAPE=4.52%  WMAPE=4.51%
Fold 5: MAE=134.0229  RMSE=156.8170  sMAPE=13.40%  WMAPE=12.85%

Cross-validation complete.


In [13]:
results_df = pd.DataFrame(fold_results).set_index('Fold')
print(results_df)
print('\n--- Mean across folds ---')
print(results_df.mean().round(4))

           MAE      RMSE  sMAPE  WMAPE
Fold                                  
1      98.0858  115.2675  14.00  13.41
2      12.5276   15.0148   1.57   1.56
3      19.6076   22.3750   2.68   2.65
4      39.0137   48.6436   4.52   4.51
5     134.0229  156.8170  13.40  12.85

--- Mean across folds ---
MAE      60.6515
RMSE     71.6236
sMAPE     7.2340
WMAPE     6.9960
dtype: float64


In [14]:
model_path = MODELS_DIR + 'xgboost_defense_market.pkl'
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

Model saved to: ../models/xgboost_defense_market.pkl


## Performance Assessment

**Data basis:** 287 observations, 32 features, target FDEFX in USD billions (SAAR).

| Fold | MAE (USD bn) | RMSE (USD bn) | sMAPE | WMAPE |
|------|-------------|--------------|-------|-------|
| 1 | 98.1 | 115.3 | 14.00 % | 13.41 % |
| 2 | 12.5 | 15.0 | 1.57 % | 1.56 % |
| 3 | 19.6 | 22.4 | 2.68 % | 2.65 % |
| 4 | 39.0 | 48.6 | 4.52 % | 4.51 % |
| 5 | 134.0 | 156.8 | 13.40 % | 12.85 % |
| **Mean** | **60.7** | **71.6** | **7.23 %** | **7.00 %** |

### Feature Reduction vs. Previous Model (87 features)

| Change | Before | After | Reason |
|--------|--------|-------|--------|
| ADEFNO absolute lags | 24 (lags 1–24) | 7 (lags 1,3,6,9,12,18,24) | Consecutive lags are highly multicollinear; sparse selection preserves 3–24 month coverage |
| ADEFNO diff lags | 24 (lags 1–24) | 2 (lags 1,3) | Only short-term momentum is informative; SHAP showed all others near zero |
| IPB52300S diff lags | 12 (lags 1–12) | 6 (lags 1–6) | Lags 7–12 showed negligible SHAP importance |
| FDEFX diff lags | 6 (lags 1–6) | 0 | Raw FDEFX_diff already present; individual diff lags added no SHAP weight |
| Rolling std features | 3 (3m/6m/12m) | 0 | Not in top-15 SHAP; FDEFX is too smooth for std to be informative |
| `year` feature | present | removed | Numeric year causes out-of-distribution extrapolation for 2026 |

**Row-to-feature ratio:** 287 / 32 ≈ **9.0 : 1** (vs. 3.3 : 1 before). Mean sMAPE improved from 7.58 % to **7.23 %**.

### Regularisation

`colsample_bytree=0.6` and `reg_alpha=0.1` added:
- Column subsampling forces trees to learn from feature subsets, preventing dominant lag features from monopolising every split.
- L1 penalty drives marginally useful feature weights toward zero.

### Fold Variance — Pattern and Explanation

**Folds 2–4 (sMAPE 1.6 %–4.5 %)** perform well, covering 2002–2014 where the model captures structural expenditure dynamics reliably.

**Fold 1 (sMAPE 14.0 %)** underperforms due to ~47 training samples — still below the 10:1 rule, unavoidable with TimeSeriesSplit.

**Fold 5 (sMAPE 13.4 %)** covers 2021–2025 (COVID recovery, Ukraine war, NATO ramp-up). Structural breaks not present in the 2002–2020 training window cause genuine out-of-distribution conditions. This is the most relevant fold for 2026 forecasting and defines the known error floor.

### Note on Metric Magnitude

The mean sMAPE of 7.23 % reflects FDEFX's smoothness (quarterly forward-fill), not forecasting skill. A smoother target mechanically yields lower percentage errors. Absolute error (MAE ≈ 61 USD bn against a series level of ~800–1,200 USD bn) is acceptable for a market volume proxy used in SAC scenario planning.